# lsu-pria — SLT (video + Whisper) en Google Colab

Este notebook entrena/arma el pipeline **SLT (Sign Language Translation)** usando **source2 + source3** ya extraídos en Google Drive.

Asume que tu dataset tiene esta estructura (o equivalente) bajo `ILSUT_ROOT`:
- `source2/episodes/*.avi` o `*.mp4`
- `source2/whisperx/...` (salida de Whisper/WhisperX por episodio)
- `source3/episodes/...`
- `source3/whisperx/...`

Importante:
- Este flujo **no descarga iLSU‑T** (acceso restringido). Solo usa archivos ya presentes en tu Drive.
- SLT en este repo se usa **offline** (clip/video → texto). Para “casi realtime”, usá el modo *Realtime SLT (beta)* en la web.


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Repo setup
# Option A (recommended): put the repo folder in Drive and point REPO_DIR to it.
# Option B: clone from a URL if your repo is public or you have access.
REPO_DIR = '/content/drive/MyDrive/lsu-pria'  # change if needed
GIT_URL = ''  # optional, e.g. 'https://github.com/<user>/<repo>.git'

import os
from pathlib import Path

repo_dir = Path(REPO_DIR)
if repo_dir.exists():
    %cd {REPO_DIR}
else:
    assert GIT_URL.strip(), 'Set REPO_DIR to an existing Drive folder OR set GIT_URL to clone.'
    !rm -rf /content/lsu-pria
    !git clone --depth 1 {GIT_URL} /content/lsu-pria
    %cd /content/lsu-pria


In [ ]:
# Install Python deps (repo)
!python3 -m pip install -U pip
!python3 -m pip install -r requirements.txt

# (Optional) Ensure ffmpeg exists (needed only if you convert AVI->MP4)
!apt-get -qq update
!apt-get -qq install -y ffmpeg


In [ ]:
# (Optional) Clone neccam/slt backend repo for generative training (SignJoey)
# Nota: el backend generativo requiere GPU/CUDA (Colab). En CPU igual podés entrenar el baseline proxy.
BACKEND_GIT = 'https://github.com/neccam/slt.git'
BACKEND_REPO = '/content/neccam-slt'
!rm -rf {BACKEND_REPO}
!git clone --depth 1 {BACKEND_GIT} {BACKEND_REPO}
print('Backend repo at', BACKEND_REPO)


## Generative SLT (neccam/slt) — entrenamiento y uso en demo

- El pipeline `run-whisperx-slt-pipeline` siempre entrena un **baseline local (proxy KNN)**.
- Si además querés un modelo **generativo** (SignJoey), necesitás el repo `neccam/slt` + deps extra (`torchtext`, `pyyaml`).

En Colab (GPU) esto es lo recomendado.


In [ ]:
# Extra deps required by neccam/slt backend (SignJoey)
!python3 -m pip install -U torchtext pyyaml


Luego, al correr el pipeline, agregá `--backend-repo` y `--run-backend` (usa CUDA en Colab):

```bash
python3 lsupria.py run-whisperx-slt-pipeline \
  --root "$ILSUT_ROOT" \
  --sources source2 source3 \
  --work-root "$WORK_ROOT" \
  --preset standard \
  --epochs 30 \
  --batch-size 32 \
  --device cuda \
  --backend-repo "$BACKEND_REPO" \
  --run-backend
```

Salida importante:
- `WORK_ROOT/train/external_backend_model/` (vocabs + ckpts)
- `WORK_ROOT/train/external_backend_config.yaml`

Para usarlo en la web demo: iniciá el backend con `--slt-gen-*` apuntando a esos paths.


In [ ]:
# Config — adjust these paths
ILSUT_ROOT = '/content/drive/MyDrive/iLSUT_extracted'  # debe contener source2/ y source3/

# Output work directory (artifacts, dataset export, models, reports)
WORK_ROOT = '/content/drive/MyDrive/lsu-pria/runs/colab_whisperx_slt_s2s3_e30'

# Dataset feature extraction params
SAMPLE_FPS = 6
MAX_FRAMES = 48   # recomendación para no explotar RAM/tiempo; podés subirlo luego
PREPROCESS = True

# Segment filtering (más estable para demo)
MIN_WORDS = 1
MAX_WORDS = 80
MAX_CHARS = 300
MIN_DUR_MS = 700
MAX_DUR_MS = 30000

# Training params
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
EPOCHS = 30 if DEVICE == 'cuda' else 10
BATCH_SIZE = 32 if DEVICE == 'cuda' else 8
print('DEVICE=', DEVICE, 'EPOCHS=', EPOCHS, 'BATCH_SIZE=', BATCH_SIZE)


In [ ]:
# (Optional) Convert AVI -> MP4 and delete AVI to save space
# If your episodes are already .mp4, you can skip this.
!python3 lsupria.py ilsut-convert-videos \
  --root "{ILSUT_ROOT}" \
  --sources source2 source3 \
  --input-ext .avi \
  --output-ext .mp4 \
  --tool ffmpeg \
  --quality balanced \
  --skip-existing \
  --delete-source


## Run SLT pipeline (support → subset → export → validate → train → eval)

Este comando deja artefactos en `WORK_ROOT/` (incluye `summary.json`, `eval_slt.json` y reportes).

In [ ]:
# Generative-oriented SLT pipeline from WhisperX segments (clip -> target_text).
# - Siempre entrena un baseline proxy local.
# - Si hay GPU/CUDA y BACKEND_REPO está presente, también entrena el backend generativo (SignJoey).

from pathlib import Path

# Habilitar backend solo si hay CUDA
backend_repo = BACKEND_REPO if (DEVICE == 'cuda' and Path(BACKEND_REPO).exists()) else ''

cmd = f"""python3 lsupria.py run-whisperx-slt-pipeline \
  --root "{ILSUT_ROOT}" \
  --sources source2 source3 \
  --work-root "{WORK_ROOT}" \
  --min-words {MIN_WORDS} \
  --max-words {MAX_WORDS} \
  --max-chars {MAX_CHARS} \
  --min-duration-ms {MIN_DUR_MS} \
  --max-duration-ms {MAX_DUR_MS} \
  --sample-fps {SAMPLE_FPS} \
  --max-frames {MAX_FRAMES} \
  {'--preprocess' if PREPROCESS else ''} \
  --epochs {EPOCHS} \
  --batch-size {BATCH_SIZE} \
  --device {DEVICE} \
  {'--backend-repo ' + backend_repo if backend_repo else ''} \
  {'--run-backend' if backend_repo else ''}
"""
print(cmd)
!{cmd}


## Outputs

Los paths exactos dependen del pipeline, pero típicamente vas a encontrar:
- `WORK_ROOT/**/summary.json`
- `WORK_ROOT/**/eval_slt.json`
- `WORK_ROOT/**/results/*.md`
- `WORK_ROOT/**/train/models/slt_proxy.joblib` (modelo proxy local usado por la web con `--slt-model`)


In [ ]:
import os
from pathlib import Path

root = Path(WORK_ROOT)
if root.exists():
    for p in sorted(root.rglob("*.json"))[:40]:
        print(p)
else:
    print("WORK_ROOT not found:", root)


## Usar el modelo SLT en la web

En tu máquina local (no en Colab), podés correr la demo con el **modelo proxy** (rápido/reproducible):
```bash
python3 lsupria.py web --slt-model /path/to/slt_proxy.joblib
```

Si además entrenaste el **generativo** (SignJoey) en Colab (CUDA), para usarlo en análisis offline:
```bash
python3 lsupria.py web \
  --slt-model /path/to/slt_proxy.joblib \
  --slt-gen-backend-repo /abs/path/to/neccam-slt \
  --slt-gen-config /path/to/external_backend_config.yaml \
  --slt-gen-ckpt /path/to/external_backend_model/<ALGUNO>.ckpt \
  --slt-gen-model-dir /path/to/external_backend_model
```

En la UI podés usar:
- **Offline video analysis** con `pipeline=slt` (proxy) o `pipeline=slt_gen` (generativo),
- **Realtime SLT (beta)** (ventana móvil) si querés algo “casi realtime” (usa el proxy).
